In [28]:
# ================================================================
# MSc Data Preprocessing - Week 7 Lab
# Model Training Time and Accuracy WITHOUT detailed preprocessing
# 
# Goal:
# - Use ALL columns as input features except target column "purchased"
# - Preprocessing performed:
#     1. Handle missing values
#     2. Convert text/category/date columns to numeric using One-Hot Encoding
# - Train 4 ML models
# - Measure training time
# - Show accuracy
# ================================================================


In [29]:
import time
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier


# ------------------------------------------------
# 1. Read dataset
# ------------------------------------------------
DATA_PATH = r"D:\datasets\bits\ecommerce_email_marketing_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully")
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())



Dataset loaded successfully
Dataset shape: (10080, 24)

Columns:
['customer_id', 'customer_name', 'email', 'city', 'state', 'age', 'gender', 'signup_date', 'last_purchase_date', 'preferred_category', 'total_orders', 'avg_order_value', 'lifetime_value', 'days_since_last_purchase', 'email_open_rate', 'click_rate', 'campaign_type', 'discount_offered', 'device_type', 'marketing_channel', 'customer_segment', 'cart_value', 'previous_returns', 'purchased']


In [30]:
# ------------------------------------------------
# 2. Define target column
# ------------------------------------------------
TARGET_COLUMN = "purchased"

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' not found in dataset.")

# X contains ALL columns except target
# This includes customer_id, customer_name, email, dates, city, category, etc.
# This is intentional because the task says: including all columns.
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("\nInput feature shape:", X.shape)
print("Target shape:", y.shape)



Input feature shape: (10080, 23)
Target shape: (10080,)


In [31]:
# ------------------------------------------------
# 3. Data preprocessing required only for program to run
# ------------------------------------------------
# ML models cannot directly understand text values like city, gender, email,
# campaign_type, dates, etc.
# Therefore, we do the minimum conversion:
# - Numeric columns: fill missing values using median
# - Non-numeric columns: fill missing values using "Missing" and apply One-Hot Encoding

numeric_columns = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_columns = X.select_dtypes(exclude=["int64", "float64"]).columns.tolist()

print("\nNumeric columns:", numeric_columns)
print("\nCategorical/Text/Date columns:", categorical_columns)

# Compatibility for different sklearn versions
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", one_hot_encoder)
])

minimum_processing = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_columns),
        ("categorical", categorical_transformer, categorical_columns)
    ],
    remainder="drop"
)




Numeric columns: ['customer_id', 'age', 'total_orders', 'avg_order_value', 'lifetime_value', 'days_since_last_purchase', 'email_open_rate', 'click_rate', 'discount_offered', 'cart_value', 'previous_returns']

Categorical/Text/Date columns: ['customer_name', 'email', 'city', 'state', 'gender', 'signup_date', 'last_purchase_date', 'preferred_category', 'campaign_type', 'device_type', 'marketing_channel', 'customer_segment']


In [32]:
# ------------------------------------------------
# 4. Train-test split
# ------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])



Training rows: 8064
Testing rows: 2016


In [33]:
# ------------------------------------------------
# 5. Define ML models
# ------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "SGD Classifier": SGDClassifier(random_state=42)
}


In [34]:
# ------------------------------------------------
# 6. Train models, measure training time, and calculate accuracy
# ------------------------------------------------
results = []

print("\n================ MODEL TRAINING RESULTS ================\n")

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("minimum_processing", minimum_processing),
        ("model", model)
    ])

    start_time = time.time()
    pipeline.fit(X_train, y_train)
    end_time = time.time()

    training_time = end_time - start_time

    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    results.append({
        "Model": model_name,
        "Training Time Seconds": round(training_time, 4),
        "Accuracy": round(accuracy, 4),
        "Accuracy Percent": round(accuracy * 100, 2)
    })

    print(f"Model: {model_name}")
    print(f"Training Time: {training_time:.4f} seconds")
    print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
    print("-" * 55)



================ MODEL TRAINING RESULTS ================

Model: Logistic Regression
Training Time: 9.5820 seconds
Accuracy: 0.8085 (80.85%)
-------------------------------------------------------
Model: Decision Tree
Training Time: 3.6872 seconds
Accuracy: 0.7679 (76.79%)
-------------------------------------------------------
Model: Random Forest
Training Time: 10.1721 seconds
Accuracy: 0.7961 (79.61%)
-------------------------------------------------------
Model: Extra Trees
Training Time: 12.1645 seconds
Accuracy: 0.7346 (73.46%)
-------------------------------------------------------
Model: SGD Classifier
Training Time: 1.1092 seconds
Accuracy: 0.6577 (65.77%)
-------------------------------------------------------


In [35]:
# ------------------------------------------------
# 7. Show results as table
# ------------------------------------------------
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Accuracy", ascending=False)

print("\nFinal Comparison Table:")
print(results_df.to_string(index=False))


# ------------------------------------------------
# 8. Save results
# ------------------------------------------------
OUTPUT_FILE = r"D:\datasets\bits\model_training_time_accuracy_without_preprocessing.csv"
results_df.to_csv(OUTPUT_FILE, index=False)

print("\nResults saved to:")
print(OUTPUT_FILE)


# ------------------------------------------------
# 9. Optional: detailed report for best model
# ------------------------------------------------
best_model_name = results_df.iloc[0]["Model"]
print("\nBest Model:", best_model_name)

print("\nImportant Teaching Note:")
print("This program uses all columns and performs only minimum conversion required for ML.")
print("Accuracy may be misleading because fields like customer_id, email, and name may cause data leakage or overfitting.")
print("In the next lab, students should perform proper preprocessing, feature selection, date handling, encoding strategy, and leakage removal.")



Final Comparison Table:
              Model  Training Time Seconds  Accuracy  Accuracy Percent
Logistic Regression                 9.5820    0.8085             80.85
      Random Forest                10.1721    0.7961             79.61
      Decision Tree                 3.6872    0.7679             76.79
        Extra Trees                12.1645    0.7346             73.46
     SGD Classifier                 1.1092    0.6577             65.77

Results saved to:
D:\datasets\bits\model_training_time_accuracy_without_preprocessing.csv

Best Model: Logistic Regression

Important Teaching Note:
This program uses all columns and performs only minimum conversion required for ML.
Accuracy may be misleading because fields like customer_id, email, and name may cause data leakage or overfitting.
In the next lab, students should perform proper preprocessing, feature selection, date handling, encoding strategy, and leakage removal.


In [11]:
## ----------------------

In [12]:
"""
MSc Data Preprocessing - Week 7 Integrated Lab
Ecommerce Email Marketing Dataset

This program connects theory from Week 3 to Week 7:
Week 3: Data Quality and Issues
Week 4: Data Cleaning
Week 5: Data Transformation and Aggregation
Week 6: Data Reduction
Week 7: Proximity Measures + Machine Learning

The program will:
1. Read dataset from D:\\datasets\\bits\\ecommerce_email_marketing_dataset.csv
2. Check and validate preprocessing problems
3. Fix preprocessing problems step by step
4. Demonstrate proximity measures: Euclidean, Manhattan, Cosine
5. Train the same ML models after preprocessing
6. Show training time and performance table
7. Save output CSV reports

Target column: purchased
"""

'\nMSc Data Preprocessing - Week 7 Integrated Lab\nEcommerce Email Marketing Dataset\n\nThis program connects theory from Week 3 to Week 7:\nWeek 3: Data Quality and Issues\nWeek 4: Data Cleaning\nWeek 5: Data Transformation and Aggregation\nWeek 6: Data Reduction\nWeek 7: Proximity Measures + Machine Learning\n\nThe program will:\n1. Read dataset from D:\\datasets\\bits\\ecommerce_email_marketing_dataset.csv\n2. Check and validate preprocessing problems\n3. Fix preprocessing problems step by step\n4. Demonstrate proximity measures: Euclidean, Manhattan, Cosine\n5. Train the same ML models after preprocessing\n6. Show training time and performance table\n7. Save output CSV reports\n\nTarget column: purchased\n'

In [36]:
import os
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

# =====================================================================
# STEP 0: FILE PATH CONFIGURATION
# =====================================================================
# On your Windows system, keep the CSV file here:
# D:\dataset\bits\ecommerce_email_marketing_dataset.csv
DATA_PATH = r"D:\datasets\bits\ecommerce_email_marketing_dataset.csv"
OUTPUT_FOLDER = r"D:\datasets\bits"

TARGET_COLUMN = "purchased"
RANDOM_STATE = 42

Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

# This list will store every preprocessing check and action.
validation_report = []


In [38]:
def add_report(theory_week, problem, columns, before, after, action_taken):
    """Store one row in the preprocessing validation report."""
    validation_report.append(
        {
            "Theory Week": theory_week,
            "Problem": problem,
            "Columns Checked": columns,
            "Before": before,
            "After": after,
            "Action Taken": action_taken,
        }
    )


def clean_column_names(dataframe):
    """Convert column names to lowercase snake_case."""
    dataframe = dataframe.copy()
    dataframe.columns = (
        dataframe.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
    )
    return dataframe


def find_existing_columns(dataframe, possible_names):
    """Return columns that exist from a possible-name list."""
    return [col for col in possible_names if col in dataframe.columns]


def make_onehot_encoder():
    """Handle different sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


In [39]:
# =====================================================================
# STEP 1: LOAD DATASET
# =====================================================================
print("\nReading dataset from:", DATA_PATH)
df = pd.read_csv(DATA_PATH)
print("Original shape:", df.shape)
print("Original columns:", list(df.columns))

# Preserve a raw copy for comparison.
raw_df = df.copy()

# =====================================================================
# WEEK 3: DATA QUALITY AND ISSUES
# =====================================================================
# The first job in data preprocessing is not to clean immediately.
# First we inspect and identify the problems.

# ---------------------------------------------------------------------
# STEP 2: STANDARDIZE COLUMN NAMES
# ---------------------------------------------------------------------
# Problem:
# Column names may have spaces, uppercase letters, or symbols.
# Why it is a problem:
# Python code becomes difficult if one column is written as "Open Rate",
# another as "openrate", and another as "OpenRate".
# Fix:
# Convert all column names to lowercase snake_case.
old_columns = list(df.columns)
df = clean_column_names(df)
add_report(
    "Week 3 - Data Quality",
    "Inconsistent column names",
    "All columns",
    old_columns,
    list(df.columns),
    "Converted column names to lowercase snake_case",
)

# Normalize target column name if needed.
# Example: Purchased, purchase, bought can be aligned manually here.
if TARGET_COLUMN not in df.columns:
    possible_target_names = ["purchase", "is_purchased", "bought", "converted", "conversion"]
    found_targets = find_existing_columns(df, possible_target_names)
    if found_targets:
        df = df.rename(columns={found_targets[0]: TARGET_COLUMN})
    else:
        raise ValueError(f"Target column '{TARGET_COLUMN}' not found.")



Reading dataset from: D:\datasets\bits\ecommerce_email_marketing_dataset.csv
Original shape: (10080, 24)
Original columns: ['customer_id', 'customer_name', 'email', 'city', 'state', 'age', 'gender', 'signup_date', 'last_purchase_date', 'preferred_category', 'total_orders', 'avg_order_value', 'lifetime_value', 'days_since_last_purchase', 'email_open_rate', 'click_rate', 'campaign_type', 'discount_offered', 'device_type', 'marketing_channel', 'customer_segment', 'cart_value', 'previous_returns', 'purchased']


In [40]:
# ---------------------------------------------------------------------
# STEP 3: EMPTY STRINGS AND TEXTUAL MISSING VALUES
# ---------------------------------------------------------------------
# Problem:
# Empty strings, spaces, "Unknown", "NA", "null" are not always treated as NaN.
# Why it is a problem:
# Missing value checks may show false results.
# Fix:
# Replace these values with np.nan.
missing_before = int(df.isna().sum().sum())
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.replace(["Unknown", "unknown", "UNKNOWN", "NA", "N/A", "null", "NULL", "None", "none"], np.nan)
missing_after = int(df.isna().sum().sum())
add_report(
    "Week 3 - Data Quality",
    "Empty strings treated as real values",
    "Various columns",
    missing_before,
    missing_after,
    "Replaced empty strings and textual missing indicators with NaN",
)

# ---------------------------------------------------------------------
# STEP 4: DUPLICATE RECORDS
# ---------------------------------------------------------------------
# Problem:
# The same customer/transaction may appear more than once.
# Why it is a problem:
# Duplicate rows can bias the model by giving repeated examples extra weight.
# Fix:
# Remove exact duplicate rows.
duplicates_before = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
duplicates_after = int(df.duplicated().sum())
add_report(
    "Week 3/4 - Quality + Cleaning",
    "Duplicate records",
    "Entire rows",
    duplicates_before,
    duplicates_after,
    "Removed exact duplicate rows",
)


In [41]:
# ---------------------------------------------------------------------
# STEP 5: EXTRA SPACES IN TEXT COLUMNS
# ---------------------------------------------------------------------
# Problem:
# "Hyderabad" and " Hyderabad " are treated as different values.
# Why it is a problem:
# It creates fake categories.
# Fix:
# Strip spaces from all text columns.
text_cols = df.select_dtypes(include=["object"]).columns.tolist()
space_issue_before = {}
for col in text_cols:
    space_issue_before[col] = int(df[col].astype(str).str.match(r"^\s+|\s+$", na=False).sum())
    df[col] = df[col].astype(str).str.strip().replace("nan", np.nan)
space_issue_after = {}
for col in text_cols:
    space_issue_after[col] = int(df[col].astype(str).str.match(r"^\s+|\s+$", na=False).sum())
add_report(
    "Week 4 - Data Cleaning",
    "Extra spaces",
    text_cols,
    space_issue_before,
    space_issue_after,
    "Applied strip() to all text columns",
)

# ---------------------------------------------------------------------
# STEP 6: MIXED LETTER CASE IN CATEGORICAL COLUMNS
# ---------------------------------------------------------------------
# Problem:
# Gmail, gmail, GMAIL are treated as separate categories.
# Fix:
# Convert selected categorical columns to consistent case.
case_columns = find_existing_columns(
    df,
    [
        "city",
        "country",
        "state",
        "device_type",
        "devicetype",
        "campaign_type",
        "campaigntype",
        "email_provider",
        "emailprovider",
        "marketing_channel",
        "preferred_category",
        "customer_segment",
    ],
)
case_before = {col: sorted(df[col].dropna().astype(str).unique())[:10] for col in case_columns}
for col in case_columns:
    # email provider/domain is more natural in lowercase; others in title case.
    if "email" in col or "provider" in col:
        df[col] = df[col].astype(str).str.strip().str.lower().replace("nan", np.nan)
    else:
        df[col] = df[col].astype(str).str.strip().str.title().replace("Nan", np.nan)
case_after = {col: sorted(df[col].dropna().astype(str).unique())[:10] for col in case_columns}
add_report(
    "Week 4 - Data Cleaning",
    "Mixed letter cases",
    case_columns,
    case_before,
    case_after,
    "Standardized text case for categorical columns",
)


In [42]:
# ---------------------------------------------------------------------
# STEP 7: INCONSISTENT GENDER VALUES
# ---------------------------------------------------------------------
# Problem:
# Male, male, M, FEMALE, F create separate categories.
# Fix:
# Map all known variants to Male, Female, Other.
gender_cols = find_existing_columns(df, ["gender"])
if gender_cols:
    gender_col = gender_cols[0]
    gender_before = df[gender_col].value_counts(dropna=False).to_dict()

    def clean_gender(value):
        if pd.isna(value):
            return np.nan
        value = str(value).strip().lower()
        if value in ["m", "male", "man"]:
            return "Male"
        if value in ["f", "female", "woman"]:
            return "Female"
        if value in ["o", "other", "others"]:
            return "Other"
        return np.nan

    df[gender_col] = df[gender_col].apply(clean_gender)
    gender_after = df[gender_col].value_counts(dropna=False).to_dict()
    add_report(
        "Week 4 - Data Cleaning",
        "Inconsistent categorical values",
        gender_col,
        gender_before,
        gender_after,
        "Mapped gender values to Male/Female/Other",
    )

# ---------------------------------------------------------------------
# STEP 8: INVALID EMAIL DOMAINS / INVALID EMAIL FORMAT
# ---------------------------------------------------------------------
# Problem:
# Invalid emails reduce customer data quality.
# Fix:
# Validate email, create is_valid_email, and extract email_domain.
if "email" in df.columns:
    email_regex = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
    invalid_email_before = int((~df["email"].astype(str).str.match(email_regex, na=False)).sum())
    df["is_valid_email"] = df["email"].astype(str).str.match(email_regex, na=False).astype(int)
    df["email_domain"] = np.where(
        df["is_valid_email"] == 1,
        df["email"].astype(str).str.split("@").str[-1].str.lower(),
        "invalid_email",
    )
    invalid_email_after = int((df["email_domain"] == "invalid_email").sum())
    add_report(
        "Week 3/4 - Quality + Cleaning",
        "Invalid email domains",
        "email",
        invalid_email_before,
        invalid_email_after,
        "Validated email format and created email_domain + is_valid_email",
    )

In [19]:
# =====================================================================
# WEEK 5: DATA TRANSFORMATION AND AGGREGATION
# =====================================================================

# ---------------------------------------------------------------------
# STEP 9: MIXED DATA TYPES IN NUMERIC COLUMNS
# ---------------------------------------------------------------------
# Problem:
# Numeric columns may contain text like "Unknown".
# Fix:
# Convert expected numeric columns to numeric. Invalid text becomes NaN.
expected_numeric_cols = find_existing_columns(
    df,
    [
        "age",
        "income",
        "lastpurchasedays",
        "last_purchase_days",
        "days_since_last_purchase",
        "openrate",
        "open_rate",
        "email_open_rate",
        "clickrate",
        "click_rate",
        "numberoforders",
        "number_of_orders",
        "total_orders",
        "totalspent",
        "total_spent",
        "lifetime_value",
        "avg_order_value",
        "cart_value",
        "discount_offered",
        "previous_returns",
        "purchasedamount",
        "purchased_amount",
    ],
)
for col in expected_numeric_cols:
    before_dtype = str(df[col].dtype)
    df[col] = pd.to_numeric(df[col], errors="coerce")
    after_dtype = str(df[col].dtype)
    add_report(
        "Week 5 - Transformation",
        "Mixed data types",
        col,
        before_dtype,
        after_dtype,
        "Converted column to numeric using errors='coerce'",
    )


In [20]:
# ---------------------------------------------------------------------
# STEP 10: INVALID NUMERIC VALUES
# ---------------------------------------------------------------------
# Problem:
# Age cannot be negative; open/click rate should be 0 to 1; spending cannot be negative.
# Fix:
# Replace invalid values with NaN before imputation.
if "age" in df.columns:
    invalid_age_before = int(((df["age"] < 0) | (df["age"] > 100)).sum())
    df.loc[(df["age"] < 0) | (df["age"] > 100), "age"] = np.nan
    invalid_age_after = int(((df["age"] < 0) | (df["age"] > 100)).sum())
    add_report(
        "Week 3/4 - Quality + Cleaning",
        "Invalid age values",
        "age",
        invalid_age_before,
        invalid_age_after,
        "Replaced age < 0 and age > 100 with NaN",
    )

rate_cols = find_existing_columns(df, ["openrate", "open_rate", "email_open_rate", "clickrate", "click_rate"])
for col in rate_cols:
    invalid_rate_before = int(((df[col] < 0) | (df[col] > 1)).sum())
    df.loc[(df[col] < 0) | (df[col] > 1), col] = np.nan
    invalid_rate_after = int(((df[col] < 0) | (df[col] > 1)).sum())
    add_report(
        "Week 3/4 - Quality + Cleaning",
        "Invalid rate values",
        col,
        invalid_rate_before,
        invalid_rate_after,
        "Replaced values outside 0 to 1 range with NaN",
    )

non_negative_cols = find_existing_columns(
    df,
    [
        "income",
        "numberoforders",
        "number_of_orders",
        "total_orders",
        "totalspent",
        "total_spent",
        "lifetime_value",
        "avg_order_value",
        "cart_value",
        "discount_offered",
        "previous_returns",
        "purchasedamount",
        "purchased_amount",
    ],
)
for col in non_negative_cols:
    negative_before = int((df[col] < 0).sum())
    df.loc[df[col] < 0, col] = np.nan
    negative_after = int((df[col] < 0).sum())
    add_report(
        "Week 3/4 - Quality + Cleaning",
        "Negative values",
        col,
        negative_before,
        negative_after,
        "Replaced negative values with NaN",
    )


In [21]:
# ---------------------------------------------------------------------
# STEP 11: DATE COLUMNS STORED AS TEXT
# ---------------------------------------------------------------------
# Problem:
# ML models cannot use raw date strings directly.
# Fix:
# Convert dates to datetime, then extract numeric features.
date_cols = find_existing_columns(
    df,
    ["signupdate", "signup_date", "lastemailsentdate", "last_email_sent_date", "last_purchase_date"],
)
for col in date_cols:
    invalid_dates_before = int(pd.to_datetime(df[col], errors="coerce", dayfirst=True).isna().sum())
    df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
    invalid_dates_after = int(df[col].isna().sum())
    df[f"{col}_year"] = df[col].dt.year
    df[f"{col}_month"] = df[col].dt.month
    df[f"{col}_dayofweek"] = df[col].dt.dayofweek
    today = pd.Timestamp.today().normalize()
    df[f"days_since_{col}"] = (today - df[col]).dt.days
    add_report(
        "Week 5 - Transformation",
        "Date stored as text",
        col,
        invalid_dates_before,
        invalid_dates_after,
        "Converted to datetime and extracted year, month, weekday, days_since features",
    )

# ---------------------------------------------------------------------
# STEP 12: SPECIAL CHARACTERS IN EMAIL SUBJECT
# ---------------------------------------------------------------------
# Problem:
# Raw text has punctuation, symbols and high cardinality.
# Fix:
# Clean text and create simple numerical NLP features.
subject_cols = find_existing_columns(df, ["emailsubject", "email_subject", "subject"])
for col in subject_cols:
    df[f"{col}_clean"] = (
        df[col]
        .astype(str)
        .str.lower()
        .str.replace(r"[^a-z0-9\s]", "", regex=True)
        .str.strip()
    )
    df[f"{col}_length"] = df[f"{col}_clean"].str.len()
    df[f"{col}_word_count"] = df[f"{col}_clean"].str.split().str.len()
    add_report(
        "Week 5 - Transformation",
        "Special characters in text",
        col,
        "Raw email subject text",
        "Cleaned text + length + word count",
        "Removed special characters and created simple NLP features",
    )

# ---------------------------------------------------------------------
# STEP 13: BOOLEAN STORED AS TEXT
# ---------------------------------------------------------------------
# Problem:
# Yes/No text values must become numeric values for ML.
# Fix:
# Convert Yes/No-like columns to 1/0.
for col in df.columns:
    if col == TARGET_COLUMN:
        continue
    if df[col].dtype == "object":
        unique_values = set(df[col].dropna().astype(str).str.lower().unique())
        allowed_boolean_values = {"yes", "no", "y", "n", "true", "false", "1", "0"}
        if len(unique_values) > 0 and unique_values.issubset(allowed_boolean_values):
            before_unique = df[col].value_counts(dropna=False).to_dict()
            df[col] = df[col].astype(str).str.lower().map(
                {
                    "yes": 1,
                    "y": 1,
                    "true": 1,
                    "1": 1,
                    "no": 0,
                    "n": 0,
                    "false": 0,
                    "0": 0,
                }
            )
            after_unique = df[col].value_counts(dropna=False).to_dict()
            add_report(
                "Week 5 - Transformation",
                "Boolean stored as text",
                col,
                before_unique,
                after_unique,
                "Mapped Yes/No-like values to 1/0",
            )


In [22]:
# ---------------------------------------------------------------------
# STEP 14: OUTLIER TREATMENT USING IQR CAPPING
# ---------------------------------------------------------------------
# Problem:
# Extreme values in income, total spent, and order count can bias the model.
# Fix:
# Cap outliers using IQR lower and upper limits.
outlier_cols = find_existing_columns(
    df,
    [
        "income",
        "totalspent",
        "total_spent",
        "lifetime_value",
        "numberoforders",
        "number_of_orders",
        "total_orders",
    ],
)
for col in outlier_cols:
    if pd.api.types.is_numeric_dtype(df[col]):
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers_before = int(((df[col] < lower) | (df[col] > upper)).sum())
        df[col] = df[col].clip(lower=lower, upper=upper)
        outliers_after = int(((df[col] < lower) | (df[col] > upper)).sum())
        add_report(
            "Week 4/5 - Cleaning + Transformation",
            "Outliers",
            col,
            outliers_before,
            outliers_after,
            "Applied IQR capping",
        )

# ---------------------------------------------------------------------
# STEP 15: HIGHLY SKEWED VARIABLES
# ---------------------------------------------------------------------
# Problem:
# Income and spending columns are often right-skewed.
# Fix:
# Create log1p features.
skew_cols = find_existing_columns(df, ["income", "totalspent", "total_spent", "lifetime_value"])
for col in skew_cols:
    if pd.api.types.is_numeric_dtype(df[col]):
        skew_before = round(float(df[col].skew(skipna=True)), 4)
        df[f"{col}_log1p"] = np.log1p(df[col].clip(lower=0))
        skew_after = round(float(df[f"{col}_log1p"].skew(skipna=True)), 4)
        add_report(
            "Week 5 - Transformation",
            "Highly skewed variables",
            col,
            skew_before,
            skew_after,
            "Created log1p transformed feature",
        )

# =====================================================================
# WEEK 6: DATA REDUCTION
# =====================================================================

# ---------------------------------------------------------------------
# STEP 16: HIGH-CARDINALITY AND IRRELEVANT COLUMNS
# ---------------------------------------------------------------------
# Problem:
# CustomerID and raw Email usually do not generalize.
# Email has too many unique values.
# Fix:
# Drop IDs and raw high-cardinality identifiers, but keep useful derived features.
columns_to_drop = []
for col in ["customerid", "customer_id", "customer_name", "email"]:
    if col in df.columns:
        columns_to_drop.append(col)

# ---------------------------------------------------------------------
# STEP 17: POSSIBLE DATA LEAKAGE COLUMNS
# ---------------------------------------------------------------------
# Problem:
# If predicting whether a customer purchased, PurchasedAmount can directly reveal the answer.
# Fix:
# Drop leakage columns.
for col in ["purchasedamount", "purchased_amount"]:
    if col in df.columns:
        columns_to_drop.append(col)

# Raw date and raw text columns were converted into numerical features, so drop raw versions.
for col in date_cols + subject_cols:
    if col in df.columns:
        columns_to_drop.append(col)
for col in [f"{c}_clean" for c in subject_cols]:
    if col in df.columns:
        columns_to_drop.append(col)

columns_to_drop = sorted(set([col for col in columns_to_drop if col != TARGET_COLUMN]))
df = df.drop(columns=columns_to_drop, errors="ignore")
add_report(
    "Week 6 - Data Reduction",
    "High-cardinality / irrelevant / leakage columns",
    columns_to_drop,
    "Columns present before reduction",
    "Dropped",
    "Removed IDs, raw email, raw dates, raw text and possible leakage columns",
)


In [23]:
# ---------------------------------------------------------------------
# STEP 18: CORRELATED FEATURES
# ---------------------------------------------------------------------
# Problem:
# Highly correlated columns such as open rate and click rate may be redundant.
# Fix:
# Report and drop one feature from pairs with correlation greater than 0.95.
# Note:
# We drop only very high correlations to avoid removing useful columns aggressively.
numeric_df_for_corr = df.select_dtypes(include=[np.number]).drop(columns=[TARGET_COLUMN], errors="ignore")
correlated_pairs = []
cols_to_drop_corr = set()
if numeric_df_for_corr.shape[1] > 1:
    corr_matrix = numeric_df_for_corr.corr().abs()
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    for column in upper_triangle.columns:
        rows = upper_triangle.index[upper_triangle[column] > 0.95].tolist()
        for row in rows:
            correlated_pairs.append((row, column, round(float(upper_triangle.loc[row, column]), 4)))
            cols_to_drop_corr.add(column)

if cols_to_drop_corr:
    df = df.drop(columns=list(cols_to_drop_corr), errors="ignore")

add_report(
    "Week 6 - Data Reduction",
    "Correlated features",
    "Numeric columns",
    correlated_pairs[:20],
    f"Dropped {len(cols_to_drop_corr)} near-duplicate columns",
    "Dropped one feature from each pair where correlation > 0.95",
)


In [24]:
# =====================================================================
# FINAL CLEANING BEFORE MODELING
# =====================================================================

# ---------------------------------------------------------------------
# STEP 19: TARGET VARIABLE CLEANING
# ---------------------------------------------------------------------
# Problem:
# Target may be stored as Yes/No or True/False.
# Fix:
# Convert target to numeric 0/1 and drop rows where target is missing.
if df[TARGET_COLUMN].dtype == "object":
    df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(str).str.lower().map(
        {
            "yes": 1,
            "y": 1,
            "true": 1,
            "1": 1,
            "no": 0,
            "n": 0,
            "false": 0,
            "0": 0,
        }
    )
df[TARGET_COLUMN] = pd.to_numeric(df[TARGET_COLUMN], errors="coerce")
rows_before_target_drop = len(df)
df = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(int)
add_report(
    "Week 3/4 - Quality + Cleaning",
    "Invalid or missing target values",
    TARGET_COLUMN,
    rows_before_target_drop,
    len(df),
    "Converted target to 0/1 and removed rows with missing target",
)

# ---------------------------------------------------------------------
# STEP 20: CLASS IMBALANCE CHECK
# ---------------------------------------------------------------------
# Problem:
# If Purchased = Yes is very rare, accuracy alone can be misleading.
# Fix:
# Use stratified split and class_weight='balanced' for supported models.
class_distribution = df[TARGET_COLUMN].value_counts(normalize=True).round(4).to_dict()
add_report(
    "Week 3 - Data Quality",
    "Class imbalance",
    TARGET_COLUMN,
    class_distribution,
    "Handled in training where supported",
    "Used stratified train-test split and class_weight='balanced' for selected models",
)

# ---------------------------------------------------------------------
# STEP 21: SEPARATE FEATURES AND TARGET
# ---------------------------------------------------------------------
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("\nShape after preprocessing:", df.shape)
print("Numeric features used:", numeric_features)
print("Categorical features used:", categorical_features)
print("Target distribution:")
print(y.value_counts())

# ---------------------------------------------------------------------
# STEP 22: MISSING VALUES + ENCODING + SCALING PIPELINE
# ---------------------------------------------------------------------
# Problem 1:
# ML models cannot handle missing values.
# Fix:
# Numeric columns -> median imputation.
# Categorical columns -> most frequent value.
#
# Problem 2:
# ML models need numerical input.
# Fix:
# One-hot encode categorical columns.
#
# Problem 3:
# Different scales distort distance-based models like KNN.
# Fix:
# StandardScaler for numeric columns.

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", make_onehot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

# =====================================================================
# WEEK 7: PROXIMITY MEASURES DEMO
# =====================================================================
# This section demonstrates why scaling matters.
# Euclidean and Manhattan distances are strongly affected by large-scale columns.
# Cosine similarity focuses more on direction/pattern.

print("\n================ WEEK 7: PROXIMITY MEASURES DEMO ================")

# Use only numeric columns for a simple distance demonstration.
proximity_features = [col for col in ["age", "income", "openrate", "open_rate", "email_open_rate", "clickrate", "click_rate"] if col in X.columns]
proximity_features = proximity_features[:4]

proximity_results = []
if len(proximity_features) >= 2 and len(X) >= 2:
    sample_numeric = X[proximity_features].copy()
    sample_numeric = sample_numeric.apply(pd.to_numeric, errors="coerce")
    sample_numeric = sample_numeric.fillna(sample_numeric.median())

    customer_a = sample_numeric.iloc[0].values.astype(float)
    customer_b = sample_numeric.iloc[1].values.astype(float)

    euclidean_raw = np.sqrt(np.sum((customer_a - customer_b) ** 2))
    manhattan_raw = np.sum(np.abs(customer_a - customer_b))
    cosine_raw = np.dot(customer_a, customer_b) / ((np.linalg.norm(customer_a) * np.linalg.norm(customer_b)) + 1e-9)

    scaler_for_demo = StandardScaler()
    scaled_values = scaler_for_demo.fit_transform(sample_numeric)
    customer_a_scaled = scaled_values[0]
    customer_b_scaled = scaled_values[1]

    euclidean_scaled = np.sqrt(np.sum((customer_a_scaled - customer_b_scaled) ** 2))
    manhattan_scaled = np.sum(np.abs(customer_a_scaled - customer_b_scaled))
    cosine_scaled = np.dot(customer_a_scaled, customer_b_scaled) / (
        (np.linalg.norm(customer_a_scaled) * np.linalg.norm(customer_b_scaled)) + 1e-9
    )

    proximity_results = [
        {"Distance Measure": "Euclidean", "Before Scaling": round(euclidean_raw, 4), "After Scaling": round(euclidean_scaled, 4)},
        {"Distance Measure": "Manhattan", "Before Scaling": round(manhattan_raw, 4), "After Scaling": round(manhattan_scaled, 4)},
        {"Distance Measure": "Cosine Similarity", "Before Scaling": round(cosine_raw, 4), "After Scaling": round(cosine_scaled, 4)},
    ]
    proximity_df = pd.DataFrame(proximity_results)
    print("Features used for proximity demo:", proximity_features)
    print(proximity_df.to_string(index=False))
else:
    proximity_df = pd.DataFrame(
        [{"Message": "Not enough numeric columns found for proximity demo."}]
    )
    print("Not enough numeric columns found for proximity demo.")



Shape after preprocessing: (10000, 27)
Numeric features used: ['age', 'total_orders', 'avg_order_value', 'lifetime_value', 'days_since_last_purchase', 'email_open_rate', 'click_rate', 'discount_offered', 'cart_value', 'previous_returns', 'is_valid_email', 'signup_date_year', 'signup_date_month', 'signup_date_dayofweek', 'last_purchase_date_month', 'last_purchase_date_dayofweek', 'lifetime_value_log1p']
Categorical features used: ['city', 'state', 'gender', 'preferred_category', 'campaign_type', 'device_type', 'marketing_channel', 'customer_segment', 'email_domain']
Target distribution:
purchased
0    6619
1    3381
Name: count, dtype: int64

================ WEEK 7: PROXIMITY MEASURES DEMO ================
Features used for proximity demo: ['age', 'email_open_rate', 'click_rate']
 Distance Measure  Before Scaling  After Scaling
        Euclidean          7.0121         2.7193
        Manhattan          7.4120         3.3507
Cosine Similarity          0.9999        -0.4170


In [25]:
# =====================================================================
# MACHINE LEARNING MODEL TRAINING
# =====================================================================

# ---------------------------------------------------------------------
# STEP 23: TRAIN-TEST SPLIT
# ---------------------------------------------------------------------
# Stratification preserves the same class ratio in train and test data.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

# ---------------------------------------------------------------------
# STEP 24: DEFINE SAME ML MODELS AS BEFORE
# ---------------------------------------------------------------------
# At least 4 models are used. Here we use 5.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=120,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=7),
}

# ---------------------------------------------------------------------
# STEP 25: TRAIN, TIME, AND EVALUATE MODELS
# ---------------------------------------------------------------------
# For each model:
# 1. Apply preprocessing pipeline
# 2. Train the model
# 3. Measure training time
# 4. Predict on test data
# 5. Calculate accuracy, precision, recall, F1 score and ROC-AUC
results = []

for model_name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessing", preprocessor),
            ("model", model),
        ]
    )

    start_time = time.perf_counter()
    pipeline.fit(X_train, y_train)
    end_time = time.perf_counter()

    training_time = end_time - start_time
    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        try:
            y_probability = pipeline.predict_proba(X_test)[:, 1]
            roc_auc = roc_auc_score(y_test, y_probability)
        except Exception:
            roc_auc = np.nan
    else:
        roc_auc = np.nan

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    results.append(
        {
            "Model": model_name,
            "Training Time Seconds": round(training_time, 4),
            "Accuracy": round(accuracy_score(y_test, y_pred), 4),
            "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
            "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
            "F1 Score": round(f1_score(y_test, y_pred, zero_division=0), 4),
            "ROC AUC": round(roc_auc, 4) if not np.isnan(roc_auc) else np.nan,
            "True Positive": int(tp),
            "True Negative": int(tn),
            "False Positive": int(fp),
            "False Negative": int(fn),
        }
    )

results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)
validation_report_df = pd.DataFrame(validation_report)


In [26]:
# =====================================================================
# SAVE OUTPUT FILES
# =====================================================================
validation_report_path = os.path.join(OUTPUT_FOLDER, "week7_preprocessing_validation_report.csv")
model_results_path = os.path.join(OUTPUT_FOLDER, "week7_model_results_after_preprocessing.csv")
proximity_results_path = os.path.join(OUTPUT_FOLDER, "week7_proximity_measures_demo.csv")
cleaned_dataset_path = os.path.join(OUTPUT_FOLDER, "week7_cleaned_ecommerce_dataset.csv")

validation_report_df.to_csv(validation_report_path, index=False)
results_df.to_csv(model_results_path, index=False)
proximity_df.to_csv(proximity_results_path, index=False)
df.to_csv(cleaned_dataset_path, index=False)

print("\n================ PREPROCESSING VALIDATION REPORT ================")
print(validation_report_df.to_string(index=False))

print("\n================ MODEL PERFORMANCE AFTER PREPROCESSING ================")
print(results_df.to_string(index=False))

print("\n================ OUTPUT FILES SAVED ================")
print("Validation report:", validation_report_path)
print("Model results:", model_results_path)
print("Proximity demo:", proximity_results_path)
print("Cleaned dataset:", cleaned_dataset_path)

print("\nTeaching note:")
print("KNN is especially important in Week 7 because it depends on distance/proximity.")
print("Without scaling, large-value features such as income dominate distance calculation.")
print("After preprocessing and scaling, distance-based learning becomes more meaningful.")



================ PREPROCESSING VALIDATION REPORT ================
                         Theory Week                                         Problem                                                                                                                                                   Columns Checked                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    Before     